# FAISS: Local Vector Search

This notebook covers using FAISS (Facebook AI Similarity Search) for local vector storage and retrieval.

## Topics
1. Installing and using FAISS
2. Creating a local vector store
3. Adding vectors and querying
4. Comparing FAISS vs Pinecone
5. When to use each

## Setup

In [ ]:
# Install FAISS
# !pip install faiss-cpu numpy
# For GPU support: !pip install faiss-gpu

import faiss
import numpy as np
import time
import json
import os

print(f"FAISS version: {faiss.__version__}")
print(f"Available GPUs: {faiss.get_num_gpus()}")

## Basic FAISS Operations

FAISS works with numpy arrays. All vectors must be float32.

In [ ]:
# Create a simple index
dimension = 4

# IndexFlatL2: exact search using L2 (Euclidean) distance
index = faiss.IndexFlatL2(dimension)

# Create some sample vectors
vectors = np.array([
    [1.0, 0.0, 0.0, 0.0],
    [0.0, 1.0, 0.0, 0.0],
    [0.0, 0.0, 1.0, 0.0],
    [0.0, 0.0, 0.0, 1.0],
    [0.5, 0.5, 0.0, 0.0],  # Between vec1 and vec2
], dtype='float32')

# Add vectors to the index
index.add(vectors)
print(f"Index contains {index.ntotal} vectors.")

# Search
query = np.array([[0.6, 0.4, 0.0, 0.0]], dtype='float32')
distances, indices = index.search(query, k=3)

print(f"\nQuery: {query[0]}")
print(f"Nearest indices: {indices[0]}")
print(f"Distances:       {distances[0]}")

## Cosine Similarity with FAISS

FAISS's `IndexFlatIP` (inner product) gives cosine similarity when vectors are normalized.

In [ ]:
# Normalize vectors for cosine similarity
def normalize_vectors(vectors: np.ndarray) -> np.ndarray:
    """Normalize vectors to unit length for cosine similarity."""
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    return vectors / norms


# Create vectors and normalize them
vectors = np.array([
    [1.0, 2.0, 3.0, 4.0],
    [5.0, 6.0, 7.0, 8.0],
    [2.0, 1.0, 0.0, 1.0],
    [0.0, 0.0, 1.0, 2.0],
], dtype='float32')

normalized = normalize_vectors(vectors)

# IndexFlatIP for cosine similarity
cosine_index = faiss.IndexFlatIP(dimension)
cosine_index.add(normalized)

# Query
query = np.array([[1.0, 2.0, 3.0, 5.0]], dtype='float32')
query_normalized = normalize_vectors(query)

scores, indices = cosine_index.search(query_normalized, k=3)

print(f"Query: {query[0]}")
print(f"Cosine similarity scores: {scores[0]}")
print(f"Nearest indices: {indices[0]}")

## Storing Metadata

FAISS only stores vectors and IDs. Metadata must be stored separately.

In [ ]:
# Metadata store (in practice, use a database)
metadata_store = {}

documents = [
    {"id": "doc1", "text": "Machine learning fundamentals", "category": "AI"},
    {"id": "doc2", "text": "Web development with React", "category": "Web"},
    {"id": "doc3", "text": "Database design patterns", "category": "DB"},
    {"id": "doc4", "text": "Neural network architectures", "category": "AI"},
    {"id": "doc5", "text": "Cloud deployment strategies", "category": "DevOps"},
]

# Simulated embeddings (dimension=4 for demo)
np.random.seed(42)
embeddings = np.random.random((len(documents), 4)).astype('float32')
embeddings = normalize_vectors(embeddings)

# Store metadata indexed by position
for i, doc in enumerate(documents):
    metadata_store[i] = doc

# Create index
index = faiss.IndexFlatIP(4)
index.add(embeddings)

print(f"Index: {index.ntotal} vectors")
print(f"Metadata entries: {len(metadata_store)}")

In [ ]:
# Search and retrieve metadata
query = np.random.random((1, 4)).astype('float32')
query = normalize_vectors(query)

scores, indices = index.search(query, k=3)

print("Results:")
for score, idx in zip(scores[0], indices[0]):
    doc = metadata_store[idx]
    print(f"  Score: {score:.4f} | {doc['id']}: {doc['text']} ({doc['category']})")

## HNSW Index (Approximate Search)

HNSW is faster for large datasets but uses more memory.

In [ ]:
# Generate a larger dataset for benchmarking
np.random.seed(123)
n_vectors = 10000
dim = 128

large_vectors = np.random.random((n_vectors, dim)).astype('float32')
large_vectors = normalize_vectors(large_vectors)

# Exact search (FlatIP)
flat_index = faiss.IndexFlatIP(dim)
flat_index.add(large_vectors)

# HNSW search
hnsw_index = faiss.IndexHNSWFlat(dim, 32)  # 32 = graph connectivity (M)
hnsw_index.hnsw.efSearch = 64  # Search depth
hnsw_index.add(large_vectors)

# Benchmark
query = np.random.random((1, dim)).astype('float32')
query = normalize_vectors(query)

# Flat search timing
start = time.time()
for _ in range(100):
    flat_index.search(query, k=10)
flat_time = time.time() - start

# HNSW search timing
start = time.time()
for _ in range(100):
    hnsw_index.search(query, k=10)
hnsw_time = time.time() - start

print(f"Dataset: {n_vectors} vectors, {dim} dimensions")
print(f"Flat (exact) search: {flat_time:.4f}s for 100 queries")
print(f"HNSW (approximate) search: {hnsw_time:.4f}s for 100 queries")
print(f"Speedup: {flat_time / hnsw_time:.1f}x")

## IVF Index (Partitioned Search)

IVF partitions vectors into clusters for faster search.

In [ ]:
# IVF index: partition into 100 clusters
nlist = 100  # Number of Voronoi cells

# IVF requires training
quantizer = faiss.IndexFlatIP(dim)
ivf_index = faiss.IndexIVFFlat(quantizer, dim, nlist, faiss.METRIC_INNER_PRODUCT)

# Train the index (required for IVF)
ivf_index.train(large_vectors)
ivf_index.add(large_vectors)

# Set nprobe: number of clusters to search
ivf_index.nprobe = 10  # Search 10 out of 100 clusters

# Benchmark
start = time.time()
for _ in range(100):
    ivf_index.search(query, k=10)
ivf_time = time.time() - start

print(f"IVF search: {ivf_time:.4f}s for 100 queries")
print(f"vs Flat: {flat_time:.4f}s")
print(f"vs HNSW: {hnsw_time:.4f}s")

## Saving and Loading FAISS Indexes

FAISS indexes can be saved to disk and loaded later.

In [ ]:
# Save index to disk
index_path = "/tmp/faiss_demo.index"
faiss.write_index(flat_index, index_path)
print(f"Saved index to {index_path}")
print(f"File size: {os.path.getsize(index_path) / 1024:.1f} KB")

# Load index from disk
loaded_index = faiss.read_index(index_path)
print(f"Loaded index with {loaded_index.ntotal} vectors.")

# Verify it works
distances, indices = loaded_index.search(query, k=3)
print(f"Search results from loaded index: {indices[0]}")

# Clean up
os.remove(index_path)

## Metadata Filtering with FAISS

FAISS doesn't support metadata filtering natively. Two approaches:
1. **Post-filter:** Search more results, then filter by metadata
2. **Separate indexes:** One index per category

In [ ]:
# Approach 1: Post-filtering
# Search extra results, then filter

category_store = {i: doc["category"] for i, doc in enumerate(documents)}

def search_with_filter(query_vec, filter_category, k=3, oversample=10):
    """Search with metadata filter by oversampling and post-filtering."""
    # Search more results than needed
    scores, indices = index.search(query_vec, k=min(oversample, index.ntotal))
    
    # Filter by category
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:  # FAISS returns -1 for missing results
            continue
        if category_store[idx] == filter_category:
            results.append({"idx": idx, "score": score, "doc": metadata_store[idx]})
            if len(results) >= k:
                break
    return results


# Test filtered search
results = search_with_filter(query, filter_category="AI")
print("Filtered results (category=AI):")
for r in results:
    print(f"  Score: {r['score']:.4f} | {r['doc']['text']}")

## FAISS vs Pinecone: Comparison

In [ ]:
comparison = {
    "Feature": [
        "Type",
        "Hosting",
        "Cost",
        "Metadata filtering",
        "Persistence",
        "Scaling",
        "Setup complexity",
        "Privacy",
        "Best for"
    ],
    "FAISS": [
        "Library",
        "Local",
        "Free",
        "Manual (post-filter)",
        "Manual (save/load)",
        "Single machine",
        "Low",
        "Full control",
        "Prototyping, learning"
    ],
    "Pinecone": [
        "Managed service",
        "Cloud",
        "Paid (free tier)",
        "Built-in",
        "Automatic",
        "Auto-scaling",
        "Low",
        "Data in cloud",
        "Production apps"
    ]
}

# Print comparison table
print(f"{'Feature':<25} {'FAISS':<30} {'Pinecone':<30}")
print("=" * 85)
for i in range(len(comparison["Feature"])):
    print(f"{comparison['Feature'][i]:<25} {comparison['FAISS'][i]:<30} {comparison['Pinecone'][i]:<30}")

## When to Use Each

**Use FAISS when:**
- Prototyping and experimentation
- Data must stay on your machine (privacy/regulatory)
- You need maximum speed (GPU support)
- Budget is a primary concern
- You need custom index configurations

**Use Pinecone when:**
- Building production applications
- You need built-in metadata filtering
- You want automatic persistence and scaling
- Team lacks infrastructure expertise
- You need high availability (SLAs)

**Use Chroma when:**
- You want a balance between FAISS and Pinecone
- You need a simple Python API with persistence
- You're building a local-first application

## Exercises

### Exercise 1: Build a Document Search with FAISS

1. Take 20+ text documents
2. Generate embeddings (use sentence-transformers or OpenAI)
3. Build a FAISS index with HNSW
4. Implement a search function that returns document text
5. Save the index and metadata to disk

### Exercise 2: Benchmark Different Index Types

1. Generate 100K random vectors (dimension=256)
2. Build FlatL2, HNSW, and IVF indexes
3. Measure: build time, query time, memory usage
4. Plot the results

### Exercise 3: FAISS + Metadata Store

1. Use SQLite as a metadata store alongside FAISS
2. Store document text, categories, and timestamps
3. Implement filtered search that queries both stores

### Exercise 4: Real Embeddings

1. Install sentence-transformers: `pip install sentence-transformers`
2. Use `all-MiniLM-L6-v2` model for embeddings
3. Embed a real document collection
4. Build FAISS index and test semantic search quality

## Key Takeaways

- FAISS is free, fast, and runs locally — great for learning and prototyping
- HNSW is usually the best balance of speed and accuracy
- Metadata filtering must be handled manually in FAISS
- For production, consider managed solutions like Pinecone
- FAISS + a metadata database can work for small-to-medium production use cases